# Week 6: Amazon Price Predictor

## How to run (step-by-step)

**Important:** Run cells in order from top to bottom. Skipping or re-running single cells can cause `NameError`.

1. **Restart kernel** – Menu: `Kernel` → `Restart Kernel` (clears old state).
2. **Run Cell 1** – Imports (you may see a XGBoost warning; that’s OK).
3. **Run Cell 2** – Config and seeds.
4. **Run Cell 3** – Generate synthetic product data (creates `df`).
5. **Run the rest** – Either run each cell one by one, or use `Kernel` → `Run All`.

**Shortcut:** After restarting the kernel, use **`Kernel` → `Run All`** to run the whole notebook in order.

In [ ]:
import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from datetime import datetime
from dotenv import load_dotenv
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, mean_absolute_error
try:
    import xgboost as xgb
except Exception as e:
    print("⚠️ XGBoost not available (on Mac run: brew install libomp). Skipping XGBoost model.")
    xgb = None

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Hugging Face (for optional upload_to_hub)
from datasets import Dataset as HFDataset

# OpenRouter (via OpenAI-compatible API)
from openai import OpenAI
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Gradio
import gradio as gr

load_dotenv()

In [ ]:
class Config:
    BASE_MODEL = "openai/gpt-4o-mini"
    OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
    DATA_SIZE = 500
    PRICE_BINS = [0, 25, 50, 100, 250, 500, 1000]
    PRICE_LABELS = ['budget', 'economy', 'mid', 'premium', 'luxury', 'ultra']
    RANDOM_SEED = 42
    
config = Config()
np.random.seed(config.RANDOM_SEED)
torch.manual_seed(config.RANDOM_SEED)
os.makedirs("./data", exist_ok=True)

In [ ]:
# Synthetic product data (same structure as Amazon-style data)
print("📦 Generating synthetic product data...")

CATEGORIES = ["Beauty", "Electronics", "Clothing", "Home", "Sports"]
BRANDS = ["BrandA", "BrandB", "BrandC", "BrandD", "Generic", "Premium Co", "EcoLife", "TechPro"]
TITLE_TEMPLATES = {
    "Beauty": ["Face Cream", "Lipstick Set", "Sunscreen SPF 50", "Hair Serum", "Body Lotion", "Eye Shadow Palette", "Perfume Roll-On", "Cleansing Oil"],
    "Electronics": ["USB-C Cable", "Wireless Earbuds", "Phone Stand", "Power Bank", "Screen Protector", "Smart Watch", "Tablet Case", "Keyboard"],
    "Clothing": ["Cotton T-Shirt", "Denim Jacket", "Running Shorts", "Winter Scarf", "Canvas Sneakers", "Polo Shirt", "Yoga Pants", "Baseball Cap"],
    "Home": ["Desk Lamp", "Throw Pillow", "Storage Bins", "Kitchen Scale", "Coaster Set", "Plant Pot", "Blanket", "Wall Hook"],
    "Sports": ["Resistance Bands", "Water Bottle", "Gym Bag", "Jump Rope", "Yoga Mat", "Dumbbells Set", "Running Belt", "Foam Roller"],
}

n = config.DATA_SIZE
np.random.seed(config.RANDOM_SEED)

data = []
for i in range(n):
    cat = np.random.choice(CATEGORIES)
    title = np.random.choice(TITLE_TEMPLATES[cat]) + f" #{i % 100}"
    price = float(np.clip(np.random.lognormal(3, 1.2), 5, 950))
    desc = f"Quality {title.lower()}. Great for daily use. Durable and reliable. Category: {cat}."
    data.append({
        "title": title,
        "description": desc,
        "price": price,
        "category": cat,
        "brand": np.random.choice(BRANDS),
        "rating": float(np.clip(np.random.normal(4.2, 0.6), 1.0, 5.0)),
    })

df = pd.DataFrame(data)
df["price_category"] = pd.cut(df["price"], bins=config.PRICE_BINS, labels=config.PRICE_LABELS, right=False)
df["text"] = df["title"] + " " + df["description"] + " " + df["category"] + " " + df["brand"]

print(f"✅ Generated {len(df)} products | Price: ${df['price'].min():.0f}-${df['price'].max():.0f}")
print(df[["title", "price", "price_category"]].head())

In [ ]:
try:
    _ = len(df)
except NameError:
    raise NameError("'df' not found. Run the previous cell (📦 Generate synthetic data) first, then run all cells in order from the top.")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['price'].hist(bins=20, ax=axes[0], edgecolor='black', color='skyblue')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price ($)')

df['price_category'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='lightgreen')
axes[1].set_title('Price Categories')
axes[1].tick_params(axis='x', rotation=45)

df['rating'].hist(bins=20, ax=axes[2], edgecolor='black', color='salmon')
axes[2].set_title('Rating Distribution')
axes[2].set_xlabel('Rating')

plt.tight_layout()
os.makedirs("./data", exist_ok=True)
plt.savefig("./data/exploration.png")
plt.show()
df.to_csv("./data/amazon_products.csv", index=False)

In [ ]:
def create_prompt(row):
    return f"""Predict price category for this Amazon product:
Title: {row['title']}
Description: {row['description'][:200]}
Category: {row['category']}
Brand: {row['brand']}
Price category ({'/'.join(config.PRICE_LABELS)}):"""

training_data = []
for _, row in df.iterrows():
    training_data.append({
        "messages": [
            {"role": "user", "content": create_prompt(row)},
            {"role": "assistant", "content": row['price_category']}
        ]
    })

with open("./data/training_data.jsonl", 'w') as f:
    for ex in training_data:
        f.write(json.dumps(ex) + '\n')
print(f"✅ Prepared {len(training_data)} training examples")

In [ ]:
# Create features for traditional ML
vectorizer = TfidfVectorizer(max_features=500, stop_words='english')
X_text = vectorizer.fit_transform(df['text'])

# Encode target
le = LabelEncoder()
y = le.fit_transform(df['price_category'])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
results = {}

# Linear Regression (as regression baseline)
lr_reg = LinearRegression()
y_price = df['price'].values
X_tr, X_te, yp_tr, yp_te = train_test_split(X_text, y_price, test_size=0.2, random_state=42)
lr_reg.fit(X_tr, yp_tr)
mae = mean_absolute_error(yp_te, lr_reg.predict(X_te))
results['Linear Regression (MAE)'] = f"${mae:.2f}"
print(f"Linear Regression MAE: ${mae:.2f}")

# Classification models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}
if xgb is not None:
    models['XGBoost'] = xgb.XGBClassifier(n_estimators=100, random_state=42)

for name, model in models.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    results[name] = f"{acc:.2%}"
    print(f"{name}: {acc:.2%}")

In [ ]:
class PriceDataset(Dataset):
    def __init__(self, X, y): 
        self.X = torch.FloatTensor(X.toarray()) if hasattr(X, 'toarray') else torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class SimpleNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x): return self.net(x)

# Convert to dense for neural network
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

# Create datasets
train_dataset = PriceDataset(X_train_dense, y_train)
test_dataset = PriceDataset(X_test_dense, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# Initialize model
model = SimpleNN(X_train_dense.shape[1], len(le.classes_))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
train_losses = []
for epoch in range(20):
    model.train()
    epoch_loss = 0
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss/len(train_loader))
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/20, Loss: {epoch_loss/len(train_loader):.4f}")

# Evaluate
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for Xb, yb in test_loader:
        outputs = model(Xb)
        _, predicted = torch.max(outputs, 1)
        total += yb.size(0)
        correct += (predicted == yb).sum().item()
nn_acc = correct / total
results['Neural Network'] = f"{nn_acc:.2%}"
print(f"Neural Network Accuracy: {nn_acc:.2%}")

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(train_losses)
plt.title('Neural Network Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.savefig("./data/nn_loss.png")
plt.show()

In [ ]:
if config.OPENROUTER_API_KEY:
    client = OpenAI(api_key=config.OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)
    
    def test_model(model_name, prompt):
        try:
            resp = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0, max_tokens=10
            )
            return resp.choices[0].message.content.strip().lower()
        except Exception as e:
            print(f"Error: {e}")
            return None
    
    # Test on small sample
    test_df = df.sample(min(20, len(df)), random_state=42)
    models_to_test = [
        "openai/gpt-4o-mini", 
        "openai/gpt-3.5-turbo",
        "anthropic/claude-3-haiku"
    ]
    
    for model in models_to_test:
        correct = 0
        total = 0
        for _, row in test_df.iterrows():
            pred = test_model(model, create_prompt(row))
            if pred:
                total += 1
                if pred == row['price_category']:
                    correct += 1
        if total > 0:
            acc = correct/total
            model_name = model.split('/')[-1]
            results[f"{model_name} (0-shot)"] = f"{acc:.2%}"
            print(f"{model_name}: {acc:.2%} ({correct}/{total})")

In [ ]:
# Separate classification and regression results
class_results = {k: v for k, v in results.items() if '%' in v}
reg_result = results.get('Linear Regression (MAE)', 'N/A')

# Plot classification results
if class_results:
    plt.figure(figsize=(10, 5))
    models = list(class_results.keys())
    accs = [float(v.strip('%'))/100 for v in class_results.values()]
    colors = ['#FF9999' if '0-shot' in m else '#66B2FF' for m in models]
    bars = plt.barh(models, accs, color=colors)
    plt.xlabel('Accuracy')
    plt.title(f'Model Comparison (Regression MAE: {reg_result})')
    for bar, acc in zip(bars, accs):
        plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                f'{acc:.1%}', va='center')
    plt.tight_layout()
    plt.savefig("./data/model_comparison.png")
    plt.show()

print("\n📊 FINAL RESULTS:")
print("-" * 40)
for model, result in results.items():
    print(f"{model:30} {result}")

In [ ]:
class PricePredictorApp:
    def __init__(self):
        self.client = OpenAI(api_key=config.OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL) if config.OPENROUTER_API_KEY else None
        
    def predict(self, title, desc, category, brand):
        if not self.client: 
            return "❌ OpenRouter API key not configured. Add to .env file."
        
        prompt = f"""Predict price category for this Amazon product:
Title: {title}
Description: {desc}
Category: {category}
Brand: {brand}
Price category ({'/'.join(config.PRICE_LABELS)}):"""
        
        try:
            resp = self.client.chat.completions.create(
                model=config.BASE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0, max_tokens=10
            )
            pred = resp.choices[0].message.content.strip().lower()
            
            # Map to price range
            ranges = {
                'budget': '$0-25', 'economy': '$25-50', 'mid': '$50-100',
                'premium': '$100-250', 'luxury': '$250-500', 'ultra': '$500-1000'
            }
            price_range = ranges.get(pred, 'Unknown')
            
            return f"""### ✅ Prediction Result
**Price Category:** {pred.upper()}
**Estimated Price:** {price_range}
**Confidence:** High (based on product attributes)"""
            
        except Exception as e:
            return f"❌ Error: {str(e)}"

# Sample data for examples
sample = df.iloc[0] if len(df) > 0 else None
examples = [[sample['title'], sample['description'][:100], sample['category'], sample['brand']]] if sample is not None else []

# Create interface
if config.OPENROUTER_API_KEY:
    app = PricePredictorApp()
    iface = gr.Interface(
        fn=app.predict,
        inputs=[
            gr.Textbox(label="📦 Product Title", placeholder="Enter product title..."),
            gr.Textbox(label="📝 Description", lines=3, placeholder="Enter product description..."),
            gr.Dropdown(choices=df['category'].unique().tolist() if len(df) > 0 else ['Beauty'], 
                       label="🏷️ Category", value='Beauty'),
            gr.Textbox(label="🏢 Brand", placeholder="Enter brand name...")
        ],
        outputs=gr.Markdown(label="🎯 Result"),
        title="🛍️ Amazon Price Predictor",
        description="Predict price category from product description using AI",
        examples=examples if examples else None,
        theme="soft"
    )
    # iface.launch(share=True)  # Uncomment to launch
    print("✅ Gradio app created. Run 'iface.launch()' to start")

In [ ]:
def upload_to_hub():
    """Upload dataset to Hugging Face Hub"""
    try:
        # Convert to HF dataset
        hf_dataset = HFDataset.from_pandas(df)
        
        # Split into train/test
        train_test = hf_dataset.train_test_split(test_size=0.2, seed=42)
        
        # Upload
        username = input("Enter your Hugging Face username: ")
        dataset_name = f"{username}/amazon-price-predictor"
        train_test.push_to_hub(dataset_name)
        print(f"✅ Dataset uploaded to https://huggingface.co/datasets/{dataset_name}")
    except Exception as e:
        print(f"Error uploading: {e}")

# Uncomment to upload
# upload_to_hub()

In [ ]:
def quick_test():
    """Test with sample products"""
    test_cases = [
        {
            "title": "Apple MacBook Pro 16-inch",
            "desc": "Powerful laptop with M3 chip, 36GB RAM, 1TB SSD",
            "category": "Electronics",
            "brand": "Apple"
        },
        {
            "title": "Basic Cotton T-Shirt",
            "desc": "Comfortable 100% cotton t-shirt, machine washable",
            "category": "Clothing",
            "brand": "Hanes"
        }
    ]
    
    print("\n🔍 QUICK TEST RESULTS:")
    print("="*50)
    for tc in test_cases:
        print(f"\nTesting: {tc['title']}")
        print(predict_price(tc['title'], tc['desc'], tc['category'], tc['brand']))

# quick_test()

In [ ]:
# Save all results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_summary = {
    "timestamp": timestamp,
    "dataset_size": len(df),
    "price_range": [float(df['price'].min()), float(df['price'].max())],
    "model_results": results,
    "categories": df['category'].unique().tolist()
}

with open(f"./data/results_{timestamp}.json", 'w') as f:
    json.dump(results_summary, f, indent=2)

# Save model artifacts
joblib_dict = {}
try:
    import joblib
    joblib.dump(vectorizer, "./data/vectorizer.pkl")
    joblib.dump(le, "./data/label_encoder.pkl")
    for name, model in models.items():
        joblib.dump(model, f"./data/{name.replace(' ', '_').lower()}.pkl")
    print("✅ Models saved to ./data/")
except:
    print("⚠️ Install joblib to save models: pip install joblib")

print(f"\n✅ Results saved to ./data/results_{timestamp}.json")
print(f"\n📊 Final Summary:")
print(f"Dataset: {len(df)} products")
print(f"Price range: ${df['price'].min():.2f} - ${df['price'].max():.2f}")
if class_results:
    best = max(class_results.items(), key=lambda x: float(x[1].strip('%')))[0]
    print(f"Best model: {best}")

In [ ]:
def run_pipeline():
    """Run complete pipeline"""
    print("="*60)
    print("🏷️ AMAZON PRICE PREDICTOR - WEEK 6 PROJECT")
    print("="*60)
    
    # 1. Load data
    print("\n1️⃣ LOADING DATA")
    print("-"*40)
    # Data already loaded in previous cells
    
    # 2. Explore data
    print("\n2️⃣ DATA SUMMARY")
    print("-"*40)
    print(f"Total products: {len(df)}")
    print(f"Categories: {df['category'].nunique()}")
    print(f"Brands: {df['brand'].nunique()}")
    print(f"Price range: ${df['price'].min():.2f} - ${df['price'].max():.2f}")
    
    # 3. Model results
    print("\n3️⃣ MODEL PERFORMANCE")
    print("-"*40)
    for model, result in results.items():
        print(f"{model:30} {result}")
    
    print("\n" + "="*60)
    print("✅ PIPELINE COMPLETE!")
    print("="*60)
    
    return df

# Run the pipeline
df = run_pipeline()

print("\n📝 Next steps:")
print("1. Run 'quick_test()' to test predictions")
print("2. Launch Gradio app with 'iface.launch()'")
print("3. Upload to Hugging Face with 'upload_to_hub()'")